In [1]:
# =====================================================
# GENERATE DATA UNTUK DASHBOARD
# =====================================================

import os
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# Buat folder
os.makedirs('dashboard_data', exist_ok=True)

# =====================================================
# CEK APAKAH DATA SUDAH ADA
# =====================================================

# Cek apakah ml_data sudah ada dari notebook sebelumnya
try:
    # Jika ml_data sudah ada dari notebook
    ml_data_exists = 'ml_data' in dir()
    ketahanan_exists = 'df_ketahanan_pangan' in dir()
    komoditas_exists = 'komoditas_potensial' in dir()
    rf_exists = 'rf_model' in dir()
    
    print("✅ Data dari notebook ditemukan!")
    
except:
    ml_data_exists = False
    ketahanan_exists = False
    komoditas_exists = False
    rf_exists = False
    print("⚠️ Data dari notebook tidak ditemukan, akan dibuat ulang...")

# =====================================================
# JIKA DATA BELUM ADA, LOAD ULANG
# =====================================================

if not ml_data_exists:
    print("🔄 Membuat data dari awal...")
    
    # Load data ekspor
    file_ekspor_nilai = 'https://docs.google.com/spreadsheets/d/1OmsoM4OaU6j596EZw5VRZapQkHFp2Uss/export?format=xlsx'
    file_ekspor_berat = 'https://docs.google.com/spreadsheets/d/1VoDJY3XtDN6fWSd77eau0c0Jv7RPgpqN/export?format=xlsx'
    
    df_ekspor_nilai = pd.read_excel(file_ekspor_nilai, skiprows=3)
    df_ekspor_berat = pd.read_excel(file_ekspor_berat, skiprows=3)
    
    df_ekspor_nilai = df_ekspor_nilai.rename(columns={df_ekspor_nilai.columns[0]: 'Komoditas'})
    df_ekspor_berat = df_ekspor_berat.rename(columns={df_ekspor_berat.columns[0]: 'Komoditas'})
    
    # Load data padi
    file_produksi_padi = 'https://docs.google.com/spreadsheets/d/1DoXPBl1A40odYFCBORBr3FF4bk2YgBwl/export?format=xlsx'
    file_luas_panen_padi = 'https://docs.google.com/spreadsheets/d/1qD99-HpB4t6r0Sab-aAv4BXzQlQjPsQ3/export?format=xlsx'
    file_produksi_sayuran = 'https://docs.google.com/spreadsheets/d/1mZXFcwLP8tE0UJw8TFTTOPIALS2HE6fa/export?format=xlsx'
    file_luas_panen_sayuran = 'https://docs.google.com/spreadsheets/d/1y1uhTZrKbcKxogF4yCJjeSA5KQ-PXdYO/export?format=xlsx'
    
    df_produksi_padi = pd.read_excel(file_produksi_padi, skiprows=3)
    df_luas_panen_padi = pd.read_excel(file_luas_panen_padi, skiprows=3)
    df_produksi_sayuran = pd.read_excel(file_produksi_sayuran)
    df_luas_panen_sayuran = pd.read_excel(file_luas_panen_sayuran)
    
    # Rename
    df_produksi_padi = df_produksi_padi.rename(columns={df_produksi_padi.columns[0]: 'Provinsi'})
    df_luas_panen_padi = df_luas_panen_padi.rename(columns={df_luas_panen_padi.columns[0]: 'Provinsi'})
    df_produksi_sayuran = df_produksi_sayuran.rename(columns={df_produksi_sayuran.columns[0]: 'Provinsi'})
    df_luas_panen_sayuran = df_luas_panen_sayuran.rename(columns={df_luas_panen_sayuran.columns[0]: 'Provinsi'})
    
    # Filter Jawa
    provinsi_jawa = ['BANTEN', 'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DI YOGYAKARTA', 'JAWA TIMUR']
    
    def filter_jawa(df):
        df = df.copy()
        df['Provinsi'] = df['Provinsi'].str.upper().str.strip()
        return df[df['Provinsi'].isin(provinsi_jawa)]
    
    # Clean data
    def drop_minus(df):
        df = df.copy()
        cols_to_drop = []
        for col in df.columns:
            if col != 'Provinsi':
                if df[col].astype(str).str.contains('-', na=False).any():
                    cols_to_drop.append(col)
        return df.drop(columns=cols_to_drop)
    
    df_produksi_padi_jawa = filter_jawa(df_produksi_padi)
    df_luas_panen_padi_jawa = filter_jawa(df_luas_panen_padi)
    df_produksi_sayuran_jawa = filter_jawa(df_produksi_sayuran)
    df_luas_panen_sayuran_jawa = filter_jawa(df_luas_panen_sayuran)
    
    df_produksi_padi_clean = drop_minus(df_produksi_padi_jawa)
    df_luas_panen_padi_clean = drop_minus(df_luas_panen_padi_jawa)
    df_produksi_sayuran_clean = drop_minus(df_produksi_sayuran_jawa)
    df_luas_panen_sayuran_clean = drop_minus(df_luas_panen_sayuran_jawa)
    
    # Melt data
    def melt_bulan(df):
        bulan = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 
                 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
        cols = [c for c in bulan if c in df.columns]
        if 'Tahunan' in df.columns:
            df = df.drop(columns=['Tahunan'])
        df_melted = pd.melt(df, id_vars=['Provinsi'], value_vars=cols, var_name='Bulan', value_name='Nilai')
        return df_melted
    
    df_produksi_padi_melted = melt_bulan(df_produksi_padi_clean)
    df_luas_panen_padi_melted = melt_bulan(df_luas_panen_padi_clean)
    
    # Rename nilai
    df_produksi_padi_melted = df_produksi_padi_melted.rename(columns={'Nilai': 'Produksi_Padi_Ton'})
    df_luas_panen_padi_melted = df_luas_panen_padi_melted.rename(columns={'Nilai': 'Luas_Panen_Ha'})
    
    # Merge
    df_padi_aggregate = df_produksi_padi_melted.merge(
        df_luas_panen_padi_melted,
        on=['Provinsi', 'Bulan']
    )
    df_padi_aggregate['Produktivitas_Ton_per_Ha'] = (
        df_padi_aggregate['Produksi_Padi_Ton'] / df_padi_aggregate['Luas_Panen_Ha']
    ).replace([np.inf, -np.inf], 0).fillna(0)
    
    # Prep ekspor
    def prep_ekspor(df):
        df = df.copy()
        komoditas_col = df.columns[0]
        df = df[~df[komoditas_col].astype(str).str.lower().str.contains('jumlah', na=False)]
        bulan = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 
                 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
        cols = [c for c in bulan if c in df.columns]
        df_melted = pd.melt(df, id_vars=[komoditas_col], value_vars=cols, var_name='Bulan', value_name='Nilai')
        df_melted = df_melted.rename(columns={komoditas_col: 'Komoditas'})
        return df_melted
    
    df_ekspor_nilai_melted = prep_ekspor(df_ekspor_nilai)
    df_ekspor_berat_melted = prep_ekspor(df_ekspor_berat)
    df_ekspor_nilai_melted = df_ekspor_nilai_melted.rename(columns={'Nilai': 'Nilai_Ekspor_Juta_USD'})
    df_ekspor_berat_melted = df_ekspor_berat_melted.rename(columns={'Nilai': 'Berat_Ekspor_Ribu_Ton'})
    
    # Identifikasi komoditas potensial
    def identifikasi_komoditas(df_nilai, df_berat):
        komoditas_col_nilai = [col for col in df_nilai.columns if 'komoditas' in col.lower()][0]
        komoditas_col_berat = [col for col in df_berat.columns if 'komoditas' in col.lower()][0]
        
        total = df_nilai.groupby(komoditas_col_nilai)['Nilai_Ekspor_Juta_USD'].sum().reset_index()
        berat = df_berat.groupby(komoditas_col_berat)['Berat_Ekspor_Ribu_Ton'].sum().reset_index()
        total = total.merge(berat, left_on=komoditas_col_nilai, right_on=komoditas_col_berat, how='left')
        total = total.rename(columns={komoditas_col_nilai: 'Komoditas'})
        
        hapus = ['sarang burung', 'lainnya', 'jumlah']
        for item in hapus:
            total = total[~total['Komoditas'].str.lower().str.contains(item, na=False)]
        
        total['Nilai_Per_Ton'] = (total['Nilai_Ekspor_Juta_USD'] / total['Berat_Ekspor_Ribu_Ton']).fillna(0)
        return total
    
    komoditas_potensial = identifikasi_komoditas(df_ekspor_nilai_melted, df_ekspor_berat_melted)
    
    # Indeks ketahanan pangan
    def ketahanan_pangan(df_padi, df_sayuran):
        df_padi = df_padi.copy()
        df_sayuran = df_sayuran.copy()
        df_padi['Provinsi'] = df_padi['Provinsi'].str.upper().str.strip()
        df_sayuran['Provinsi'] = df_sayuran['Provinsi'].str.upper().str.strip()
        
        padi_agg = df_padi.groupby('Provinsi').agg({
            'Produksi_Padi_Ton': 'sum',
            'Luas_Panen_Ha': 'sum',
            'Produktivitas_Ton_per_Ha': 'mean'
        }).reset_index()
        
        cols = [col for col in df_sayuran.columns if col != 'Provinsi']
        numeric = df_sayuran[cols].select_dtypes(include=['float64', 'int64'])
        jumlah = (numeric > 0).sum(axis=1)
        
        sayuran_df = pd.DataFrame({'Provinsi': df_sayuran['Provinsi'].values, 'Jumlah_Komoditas': jumlah.values})
        security_df = padi_agg.merge(sayuran_df, on='Provinsi', how='left')
        security_df['Jumlah_Komoditas'] = security_df['Jumlah_Komoditas'].fillna(0)
        
        for col in ['Produksi_Padi_Ton', 'Produktivitas_Ton_per_Ha', 'Jumlah_Komoditas']:
            min_val = security_df[col].min()
            max_val = security_df[col].max()
            if max_val - min_val > 0:
                security_df[f'{col}_Index'] = (security_df[col] - min_val) / (max_val - min_val)
            else:
                security_df[f'{col}_Index'] = 0.5
        
        security_df['Indeks_Ketahanan_Pangan'] = security_df[
            ['Produksi_Padi_Ton_Index', 'Produktivitas_Ton_per_Ha_Index', 'Jumlah_Komoditas_Index']
        ].mean(axis=1)
        
        security_df['Tingkat_Ketahanan'] = pd.cut(
            security_df['Indeks_Ketahanan_Pangan'],
            bins=[0, 0.25, 0.5, 0.75, 1.01],
            labels=['Sangat Rendah', 'Rendah', 'Sedang', 'Tinggi'],
            right=False
        )
        return security_df
    
    df_ketahanan_pangan = ketahanan_pangan(df_padi_aggregate, df_produksi_sayuran_clean)
    
    # Prepare ML data
    def prepare_ml(df_security, df_ekspor, df_sayuran):
        df_ekspor = df_ekspor.copy()
        komoditas_col = [col for col in df_ekspor.columns if 'komoditas' in col.lower()][0]
        
        hapus = ['sarang burung', 'lainnya', 'jumlah']
        for item in hapus:
            df_ekspor = df_ekspor[~df_ekspor[komoditas_col].astype(str).str.lower().str.contains(item, na=False)]
        
        ekspor_agg = df_ekspor.groupby(komoditas_col)['Nilai_Ekspor_Juta_USD'].sum().reset_index()
        ekspor_agg = ekspor_agg.rename(columns={komoditas_col: 'Komoditas'})
        
        df_sayuran = df_sayuran.copy()
        df_sayuran['Provinsi'] = df_sayuran['Provinsi'].str.upper().str.strip()
        df_sayuran_jawa = df_sayuran[df_sayuran['Provinsi'].isin(provinsi_jawa)]
        sayuran_prov = df_sayuran_jawa.set_index('Provinsi')
        
        df_security = df_security.copy()
        df_security['Provinsi'] = df_security['Provinsi'].str.upper().str.strip()
        df_security_jawa = df_security[df_security['Provinsi'].isin(provinsi_jawa)]
        
        ml_data = df_security_jawa[['Provinsi', 'Indeks_Ketahanan_Pangan', 'Jumlah_Komoditas']].copy()
        
        cols = [col for col in sayuran_prov.columns if 'Produksi' in col]
        for komoditas in cols:
            values = []
            for prov in ml_data['Provinsi']:
                if prov in sayuran_prov.index:
                    val = sayuran_prov.loc[prov, komoditas]
                    values.append(0 if pd.isna(val) else val)
                else:
                    values.append(0)
            ml_data[komoditas] = values
        
        ml_data['Total_Ekspor_Potensial'] = ekspor_agg['Nilai_Ekspor_Juta_USD'].sum()
        median = ml_data['Indeks_Ketahanan_Pangan'].median()
        ml_data['Potensi_Ekspor'] = (ml_data['Indeks_Ketahanan_Pangan'] > median).astype(int)
        return ml_data
    
    ml_data = prepare_ml(df_ketahanan_pangan, df_ekspor_nilai_melted, df_produksi_sayuran_clean)
    
    # Train Random Forest
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score
    
    X = ml_data.drop(columns=['Potensi_Ekspor', 'Provinsi'])
    y = ml_data['Potensi_Ekspor']
    X = X.fillna(0)
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
    
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(X, y)
    
    # Feature importance
    rf_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("✅ Semua data berhasil dibuat!")

# =====================================================
# SIMPAN SEMUA DATA KE FOLDER DASHBOARD_DATA
# =====================================================

# Simpan data
ml_data.to_csv('dashboard_data/ml_data.csv', index=False)
df_ketahanan_pangan.to_csv('dashboard_data/ketahanan_pangan.csv', index=False)
komoditas_potensial.to_csv('dashboard_data/komoditas_potensial.csv', index=False)
rf_importance.to_csv('dashboard_data/feature_importance.csv', index=False)

# Simpan model
with open('dashboard_data/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("="*60)
print("✅ SEMUA DATA BERHASIL DISIMPAN!")
print("="*60)
print("\n📁 Folder: dashboard_data/")
print("📄 File yang tersimpan:")
print("   - ml_data.csv")
print("   - ketahanan_pangan.csv")
print("   - komoditas_potensial.csv")
print("   - feature_importance.csv")
print("   - rf_model.pkl")
print("\n🚀 Sekarang jalankan dashboard:")
print("   streamlit run dashboard.py")

✅ Data dari notebook ditemukan!
🔄 Membuat data dari awal...
✅ Semua data berhasil dibuat!
✅ SEMUA DATA BERHASIL DISIMPAN!

📁 Folder: dashboard_data/
📄 File yang tersimpan:
   - ml_data.csv
   - ketahanan_pangan.csv
   - komoditas_potensial.csv
   - feature_importance.csv
   - rf_model.pkl

🚀 Sekarang jalankan dashboard:
   streamlit run dashboard.py
